# Transfer Learning — AC CTRL → Reactor Ciclopentanol
Transferir el agente AC pre-entrenado (CSTR original) al reactor de Ciclopentanol.

**Objetivo:** Demostrar que el agente RL puede transferir conocimiento de ajuste PID
entre reactores CSTR con dinámicas diferentes.

**Baselines de comparación (informe 2015):**
- Ziegler-Nichols: Kp=1081.4, τI=0.020, τD=0.005
- Ajuste manual: CB→v (Kp=100, Ki=1000, Kd=0.01), T→QK (Kp=20, Ki=2000, Kd=0)

**Dos experimentos:**
1. Transfer Learning (cargar pesos del CSTR y fine-tunear)
2. From Scratch (entrenar desde cero para comparar velocidad de convergencia)

## 1. Instalación e Imports

In [ ]:
import os
import random
import numpy as np
import torch
import wandb
import sys
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Clonar desde Github:
!git clone https://github.com/valeriaeskenazi/Control-PID-Adaptativo-Inteligente-mediante-Reinforcement-Learning.git
PROJECT_PATH = '/content/Control-PID-Adaptativo-Inteligente-mediante-Reinforcement-Learning/Version_4'
sys.path.append(PROJECT_PATH)

In [ ]:
# Subir los archivos nuevos al proyecto:
# - Reactor_Cyclopentanol.py → PROJECT_PATH/Environment/Simulation_Env/
# - transfer_learning_AC.py  → PROJECT_PATH/Agente/AC/
#
# O alternativamente, subirlos a /content/ y agregar al path:
# from google.colab import files
# uploaded = files.upload()  # Subir Reactor_Cyclopentanol.py y transfer_learning_AC.py

In [ ]:
from Environment.Simulation_Env.Reactor_CSTR import CSTRSimulator
from Environment.Simulation_Env.Reactor_Cyclopentanol import CyclopentanolReactor
from Environment.PIDControlEnv_simple import PIDControlEnv_Simple
from Environment.Simulation_Env.SimulationEnv import SimulationPIDEnv
from Agente.Actor_Critic.train_AC import ACTrainer
from Agente.Actor_Critic.algorithm_AC import ACAgent
from Agente.memory import SimpleReplayBuffer
from Agente.Actor_Critic.transfer_learning_AC import setup_transfer_learning, freeze_layers
from Aux.PIDComponents_PID import PIDController
from Aux.PIDComponents_time import ResponseTimeDetector
from Aux.PIDComponentes_translate import ApplyAction

print('Imports completados')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')

## 2. Configuración

In [ ]:
# ============ FIJOS ============
SEED   = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ============ REACTOR CICLOPENTANOL ============
N_MANIPULABLE_VARS = 2
MANIPULABLE_RANGES = [(50.0, 800.0), (-8500.0, 0.0)]  # [v (L/h), QK (kJ/h)]
DT = 0.01  # horas (dinámica más rápida que el CSTR original)

# ============ HIPERPARÁMETROS DEL MEJOR AC (del sweep CSTR) ============
# IMPORTANTE: deben coincidir con el modelo pre-entrenado
HIDDEN_DIMS  = (256, 128, 64)    # del sweep ganador
LR_ACTOR     = 1e-05
LR_CRITIC    = 1e-03
GAMMA        = 0.99
ENTROPY_COEF = 0.01

# ============ CHECKPOINT DEL MODELO PRE-ENTRENADO ============
AC_CHECKPOINT = '/content/agent_ctrl_best.pt'  # ← CAMBIAR a tu ruta

# ============ REPRODUCIBILIDAD ============
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print(f'Config lista | Device: {DEVICE} | Seed: {SEED}')

## 3. Verificar Reactor Ciclopentanol

In [ ]:
# Crear reactor y verificar estado estacionario
reactor_test = CyclopentanolReactor(dt=DT, control_limits=(MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1]))
reactor_test.verify_steady_state()

# Simular 10 horas en SS para verificar estabilidad
reactor_test.reset()
for _ in range(1000):
    pvs = reactor_test.simulate_step_multi([reactor_test.v_ss, reactor_test.QK_ss], DT)
print(f'\nEstabilidad 10h: CB={pvs[0]:.4f} (SS={reactor_test.CB_ss:.4f}), T={pvs[1]:.2f} (SS={reactor_test.T_ss:.2f})')

## 4. Entrenamiento con Transfer Learning

In [ ]:
# ============ CONFIG TRANSFER LEARNING ============

WANDB_ENTITY  = 've326684-universidad-ort-uruguay'
WANDB_PROJECT = 'Tesis_AC_TransferLearning'
RUN_NAME      = 'ac_transfer_cyclopentanol'

N_EPISODES             = 5000
EVAL_FREQUENCY         = 100
LOG_FREQUENCY          = 100
SAVE_FREQUENCY         = 1000
EARLY_STOPPING_PATIENCE = 20

trainer_config_transfer = {
    'env_config': {
        'architecture'           : 'simple',
        'env_type'               : 'simulation',
        'action_type'            : 'continuous',
        'n_manipulable_vars'     : N_MANIPULABLE_VARS,
        'manipulable_ranges'     : MANIPULABLE_RANGES,
        'manipulable_setpoints'  : None,
        'dt_usuario'             : DT,
        'max_steps'              : 100,
        'max_time_detector'      : 5.0,      # 5 horas max por step
        'reward_dead_band'       : 0.02,
        'delta_percent_ctrl'     : 0.2,
        'reward_weights'         : {'error': 1.0, 'tiempo': 0.001, 'overshoot': 0.3, 'energy': 0.001},
        'pid_limits'             : [(0.01, 5000.0), (0.0, 50000.0), (0.0, 100.0)],
        'agent_controller_config': {'agent_type': 'continuous'},
        'env_type_config'        : {
            'dt': DT,
            'control_limits': (MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1])
        },
        'stability_config': {
            'error_increase_tolerance': 2.0,
            'max_sign_changes_ratio'  : 0.3,
            'max_abrupt_change_ratio' : 0.05,
            'abrupt_change_threshold' : 0.2,
        },
    },
    'agent_ctrl_config': {
        'algorithm'    : 'ac',
        'state_dim'    : N_MANIPULABLE_VARS * 5,   # 10
        'action_dim'   : N_MANIPULABLE_VARS * 3,   # 6
        'n_vars'       : N_MANIPULABLE_VARS,
        'action_type'  : 'continuous',
        'hidden_dims'  : HIDDEN_DIMS,               # (64, 32) ← DEBE coincidir
        'lr_actor'     : LR_ACTOR / 3,              # 3x menor para fine-tuning
        'lr_critic'    : LR_CRITIC / 3,             # 3x menor para fine-tuning
        'gamma'        : GAMMA,
        'entropy_coef' : ENTROPY_COEF * 2,          # Más exploración al inicio
        'batch_size'   : 64,
        'buffer_size'  : 50000,
        'warmup_steps' : 200,                        # Menos warmup (ya sabe algo)
        'device'       : DEVICE,
        'seed'         : SEED,
    },
    'n_episodes'                  : N_EPISODES,
    'eval_frequency'              : EVAL_FREQUENCY,
    'log_frequency'               : LOG_FREQUENCY,
    'save_frequency'              : SAVE_FREQUENCY,
    'checkpoint_dir'              : f'checkpoints/{RUN_NAME}',
    'early_stopping_patience'     : EARLY_STOPPING_PATIENCE,
    'early_stopping_min_delta_pct': 0.01,
    'use_wandb': True,
}

In [ ]:
# ============ CREAR REACTOR + ENVIRONMENT ============
reactor_transfer = CyclopentanolReactor(
    dt=DT,
    control_limits=(MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1])
)

# ============ CREAR TRAINER (crea el env internamente) ============
trainer_transfer = ACTrainer(trainer_config_transfer)
trainer_transfer.env.proceso.connect_external_process(reactor_transfer)

# ============ TRANSFER LEARNING: cargar pesos + congelar capas ============
agent_transfer = setup_transfer_learning(
    agent_class=ACAgent,
    checkpoint_path=AC_CHECKPOINT,
    lr_actor=LR_ACTOR / 3,
    lr_critic=LR_CRITIC / 3,
    freeze_strategy='early',     # Congelar primera capa
    n_freeze=2,                  
    entropy_coef=ENTROPY_COEF * 2,
    state_dim=10,
    action_dim=6,
    n_vars=2,
    hidden_dims=HIDDEN_DIMS,
    buffer_size=50000,
    batch_size=64,
    warmup_steps=200,
    device=DEVICE
)

# Reemplazar el agente del trainer con el pre-entrenado
trainer_transfer.agent_ctrl = agent_transfer
print('\n Agente con transfer learning conectado al trainer')

In [ ]:
# ============ INIT WANDB ============
wandb.init(
    project = WANDB_PROJECT,
    entity  = WANDB_ENTITY,
    name    = RUN_NAME,
    tags    = ['ac', 'transfer_learning', 'cyclopentanol'],
    config  = trainer_config_transfer,
)

# ============ ENTRENAR ============
trainer_transfer.train()

# ============ MÉTRICAS FINALES ============
wandb.log({
    'final_eval_reward'       : trainer_transfer.best_reward,
    'total_episodes'          : len(trainer_transfer.episode_rewards),
    'final_reward_mean10'     : np.mean(trainer_transfer.episode_rewards[-10:]),
    'final_energy_mean10'     : np.mean(trainer_transfer.episode_energies[-10:]),
    'final_overshoot_mean10'  : np.mean(trainer_transfer.episode_max_overshoots[-10:]),
}, step=len(trainer_transfer.episode_rewards))

wandb.finish()
print(f'Transfer Learning completado: {RUN_NAME}')

## 5. Entrenamiento From Scratch (Baseline)

In [ ]:
# ============ CONFIG FROM SCRATCH ============
RUN_NAME_SCRATCH = 'ac_scratch_cyclopentanol'

trainer_config_scratch = trainer_config_transfer.copy()
trainer_config_scratch['agent_ctrl_config'] = {
    'algorithm'    : 'ac',
    'state_dim'    : 10,
    'action_dim'   : 6,
    'n_vars'       : 2,
    'action_type'  : 'continuous',
    'hidden_dims'  : HIDDEN_DIMS,
    'lr_actor'     : LR_ACTOR,       # Original, sin reducir
    'lr_critic'    : LR_CRITIC,
    'gamma'        : GAMMA,
    'entropy_coef' : ENTROPY_COEF,
    'batch_size'   : 64,
    'buffer_size'  : 50000,
    'warmup_steps' : 500,            # Warmup normal
    'device'       : DEVICE,
    'seed'         : SEED,
}
trainer_config_scratch['n_episodes'] = 15000            # Más episodios
trainer_config_scratch['checkpoint_dir'] = f'checkpoints/{RUN_NAME_SCRATCH}'

# ============ CREAR Y ENTRENAR ============
reactor_scratch = CyclopentanolReactor(
    dt=DT,
    control_limits=(MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1])
)
trainer_scratch = ACTrainer(trainer_config_scratch)
trainer_scratch.env.proceso.connect_external_process(reactor_scratch)

wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name=RUN_NAME_SCRATCH,
    tags=['ac', 'from_scratch', 'cyclopentanol'],
    config=trainer_config_scratch,
)

trainer_scratch.train()

wandb.log({
    'final_eval_reward': trainer_scratch.best_reward,
    'total_episodes': len(trainer_scratch.episode_rewards),
})
wandb.finish()
print(f'From Scratch completado: {RUN_NAME_SCRATCH}')

## 6. Comparación: Transfer vs Scratch vs Baselines

In [ ]:
# ============ CURVAS DE CONVERGENCIA ============
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Reward
window = 50
r_transfer = pd.Series(trainer_transfer.episode_rewards).rolling(window).mean()
r_scratch  = pd.Series(trainer_scratch.episode_rewards).rolling(window).mean()

axes[0].plot(r_transfer, label='Transfer Learning', color='steelblue', linewidth=2)
axes[0].plot(r_scratch,  label='From Scratch', color='coral', linewidth=2)
axes[0].set_title('Reward (media móvil 50 ep)')
axes[0].set_xlabel('Episodio')
axes[0].set_ylabel('Reward')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Actor Loss
l_transfer = pd.Series(trainer_transfer.actor_losses).rolling(window).mean()
l_scratch  = pd.Series(trainer_scratch.actor_losses).rolling(window).mean()

axes[1].plot(l_transfer, label='Transfer Learning', color='steelblue', linewidth=2)
axes[1].plot(l_scratch,  label='From Scratch', color='coral', linewidth=2)
axes[1].set_title('Actor Loss (media móvil 50 ep)')
axes[1].set_xlabel('Episodio')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Transfer Learning vs From Scratch — Reactor Ciclopentanol', fontweight='bold')
plt.tight_layout()
plt.savefig('transfer_vs_scratch_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Evaluación del Mejor Agente

In [ ]:
# ============ CARGAR MEJOR AGENTE (transfer) ============
BEST_CHECKPOINT = f'checkpoints/{RUN_NAME}/agent_ctrl_best.pt'

agent_eval = ACAgent(
    state_dim=10, action_dim=6, agent_role='ctrl', n_vars=2,
    hidden_dims=HIDDEN_DIMS,
    lr_actor=LR_ACTOR, lr_critic=LR_CRITIC,
    gamma=GAMMA, entropy_coef=ENTROPY_COEF,
    batch_size=64, buffer_size=50000, warmup_steps=500,
    device=DEVICE, seed=SEED
)
agent_eval.load(BEST_CHECKPOINT)
print('Agente cargado')

In [ ]:
# ============ EVALUACIÓN MANUAL PASO A PASO ============
reactor_eval = CyclopentanolReactor(dt=DT, control_limits=(MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1]))
reactor_eval.reset()

pid_controllers = [
    PIDController(kp=1.0, ki=0.1, kd=0.01, dt=DT, output_limits=MANIPULABLE_RANGES[0]),
    PIDController(kp=1.0, ki=0.1, kd=0.01, dt=DT, output_limits=MANIPULABLE_RANGES[1]),
]
apply_action = ApplyAction(
    delta_percent_ctrl=0.2,
    pid_limits=[(0.01, 5000.0), (0.0, 50000.0), (0.0, 100.0)],
    manipulable_ranges=MANIPULABLE_RANGES
)

detector = ResponseTimeDetector(proceso=reactor_eval, env_type='simulation', dt=DT, tolerance=0.02)

# Setpoints de prueba
CB_sp, T_sp = 0.95, 410.0
sps = [CB_sp, T_sp]
pvs = reactor_eval.get_initial_pvs()  # [CB_ss, T_ss]

error_integral = [0.0, 0.0]
error_prev     = [sps[i] - pvs[i] for i in range(2)]

traj_CB, traj_T = [pvs[0]], [pvs[1]]
traj_kp0, traj_kp1 = [], []

for step in range(20):
    errors = [sps[i] - pvs[i] for i in range(2)]
    for i in range(2):
        error_integral[i] += errors[i] * DT
    error_derivative = [errors[i] - error_prev[i] for i in range(2)]
    error_prev = errors.copy()

    state = np.array([
        pvs[0], sps[0], errors[0], error_integral[0], error_derivative[0],
        pvs[1], sps[1], errors[1], error_integral[1], error_derivative[1],
    ], dtype=np.float32)

    action = agent_eval.select_action(state, training=False)

    pid_params = apply_action.translate(
        action=action, agent_type='ctrl', action_type='continuous',
        current_values=[(p.kp, p.ki, p.kd) for p in pid_controllers]
    )
    for i, (kp, ki, kd) in enumerate(pid_params):
        pid_controllers[i].kp = kp
        pid_controllers[i].ki = ki
        pid_controllers[i].kd = kd

    traj_kp0.append(pid_controllers[0].kp)
    traj_kp1.append(pid_controllers[1].kp)

    resultado = detector.estimate(
        pvs_inicial=pvs, sps=sps,
        pid_controllers=pid_controllers,
        max_time=5.0, reset_pid=False
    )

    pvs = resultado['pvs_final']
    traj_CB.append(pvs[0])
    traj_T.append(pvs[1])
    print(f'Step {step+1:2d} | CB={pvs[0]:.4f} | T={pvs[1]:.2f} | '
          f'kp_CB={pid_controllers[0].kp:.4f} | kp_T={pid_controllers[1].kp:.4f}')

In [ ]:
# ============ GRAFICAR ============
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(traj_CB, marker='o', color='steelblue', label='CB (PV)')
axes[0].axhline(CB_sp, color='red', linestyle='--', linewidth=1.5, label=f'SP={CB_sp}')
axes[0].axhspan(0.7, 1.15, alpha=0.1, color='green', label='Rango admisible')
axes[0].set_title('Concentración CB')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('CB (mol/L)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(traj_T, marker='o', color='orange', label='T (PV)')
axes[1].axhline(T_sp, color='red', linestyle='--', linewidth=1.5, label=f'SP={T_sp}K')
axes[1].set_title('Temperatura T')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('T (K)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Agente AC (Transfer Learning) — CB_sp={CB_sp}, T_sp={T_sp}K', fontweight='bold')
plt.tight_layout()
plt.savefig('eval_transfer_cyclopentanol.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Comparación con Baselines del Informe 2015

In [ ]:
# ============ BASELINE: PID MANUAL DEL INFORME ============
def run_baseline_pid(reactor, CB_sp, T_sp, Kp_CB, Ki_CB, Kd_CB, Kp_T, Ki_T, Kd_T, n_steps=2000):
    """Simula el reactor con PID fijos (baselines del informe 2015)."""
    reactor.reset()
    integral_CB, integral_T = 0.0, 0.0
    prev_err_CB, prev_err_T = 0.0, 0.0
    dt = reactor.dt
    cb_hist, t_hist = [], []
    
    for _ in range(n_steps):
        meas = reactor.get_measurements()
        
        err_CB = CB_sp - meas['CB']
        integral_CB += err_CB * dt
        deriv_CB = (err_CB - prev_err_CB) / dt
        v_out = reactor.v_ss + Kp_CB * err_CB + Ki_CB * integral_CB + Kd_CB * deriv_CB
        v_out = np.clip(v_out, 50, 800)
        prev_err_CB = err_CB
        
        err_T = T_sp - meas['T']
        integral_T += err_T * dt
        deriv_T = (err_T - prev_err_T) / dt
        QK_out = reactor.QK_ss + Kp_T * err_T + Ki_T * integral_T + Kd_T * deriv_T
        QK_out = np.clip(QK_out, -8500, 0)
        prev_err_T = err_T
        
        pvs = reactor.simulate_step_multi([v_out, QK_out], dt)
        cb_hist.append(pvs[0])
        t_hist.append(pvs[1])
    
    return cb_hist, t_hist


# ============ CORRER BASELINES ============
CB_sp, T_sp = 0.95, 410.0

reactor_bl = CyclopentanolReactor(dt=DT, control_limits=(MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1]))

# Ajuste manual (informe sección 4.3)
cb_manual, t_manual = run_baseline_pid(
    reactor_bl, CB_sp, T_sp,
    Kp_CB=100, Ki_CB=1000, Kd_CB=0.01,
    Kp_T=20, Ki_T=2000, Kd_T=0.0
)

# Ziegler-Nichols CB + manual T
cb_zn, t_zn = run_baseline_pid(
    reactor_bl, CB_sp, T_sp,
    Kp_CB=1081.4, Ki_CB=1081.4/0.020, Kd_CB=0,  # ZN sin Kd (mejor resultado del informe)
    Kp_T=20, Ki_T=2000, Kd_T=0.0
)

print(f'Manual:  CB_final={cb_manual[-1]:.4f} (err={abs(cb_manual[-1]-CB_sp):.5f}), T_final={t_manual[-1]:.2f}')
print(f'Z-N:     CB_final={cb_zn[-1]:.4f} (err={abs(cb_zn[-1]-CB_sp):.5f}), T_final={t_zn[-1]:.2f}')

In [ ]:
# ============ COMPARAR TODOS ============
time_hours = np.arange(len(cb_manual)) * DT

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# CB
axes[0].plot(time_hours, cb_manual, label='PID Manual (2015)', color='gray', linewidth=1.5)
axes[0].plot(time_hours, cb_zn, label='Ziegler-Nichols', color='orange', linewidth=1.5)
axes[0].plot(np.arange(len(traj_CB)) * (5.0),  # tiempo aprox por step
             traj_CB, label='Agente RL (Transfer)', color='steelblue', linewidth=2, marker='o')
axes[0].axhline(CB_sp, color='red', linestyle='--', linewidth=1, label=f'SP={CB_sp}')
axes[0].axhspan(0.7, 1.15, alpha=0.1, color='green')
axes[0].set_title('Concentración CB')
axes[0].set_xlabel('Tiempo (h)')
axes[0].set_ylabel('CB (mol/L)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# T
axes[1].plot(time_hours, t_manual, label='PID Manual (2015)', color='gray', linewidth=1.5)
axes[1].plot(time_hours, t_zn, label='Ziegler-Nichols', color='orange', linewidth=1.5)
axes[1].plot(np.arange(len(traj_T)) * (5.0),
             traj_T, label='Agente RL (Transfer)', color='steelblue', linewidth=2, marker='o')
axes[1].axhline(T_sp, color='red', linestyle='--', linewidth=1, label=f'SP={T_sp}K')
axes[1].set_title('Temperatura T')
axes[1].set_xlabel('Tiempo (h)')
axes[1].set_ylabel('T (K)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('RL (Transfer Learning) vs Métodos Tradicionales — Reactor Ciclopentanol', fontweight='bold')
plt.tight_layout()
plt.savefig('rl_vs_traditional_cyclopentanol.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Pruebas con otros SP (Grilla)

In [ ]:
# ============ GRILLA DE SETPOINTS ============
CB_setpoints = [0.75, 0.85, 0.95, 1.05, 1.10]  # mol/L (rango admisible: 0.7-1.15)
T_setpoints  = [405, 408, 410, 412, 415]        # K

# ============ FUNCIÓN DE EVALUACIÓN ============
eval_config = trainer_config_transfer.copy()
eval_config['n_episodes'] = 1
eval_config['use_wandb'] = False
eval_config['checkpoint_dir'] = 'checkpoints/eval_tmp'

reactor_eval2 = CyclopentanolReactor(dt=DT, control_limits=(MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1]))
trainer_eval = ACTrainer(eval_config)
trainer_eval.env.proceso.connect_external_process(reactor_eval2)
trainer_eval.agent_ctrl.load(BEST_CHECKPOINT)
print('Agente cargado para evaluación')


def evaluar_sp(trainer, CB_sp, T_sp, max_steps=100):
    state = trainer.env.reset()[0]
    trainer.env.manipulable_setpoints = [CB_sp, T_sp]
    trainer.env._update_errors()
    state = trainer.env._get_observation()

    CB_hist, T_hist = [], []
    done = False
    step = 0

    while not done and step < max_steps:
        action = trainer.agent_ctrl.select_action(state, training=False)
        next_state, reward, terminated, truncated, info = trainer.env.step(action)
        done = terminated or truncated
        CB_hist.append(trainer.env.manipulable_pvs[0])
        T_hist.append(trainer.env.manipulable_pvs[1])
        state = next_state
        step += 1

    return np.array(CB_hist), np.array(T_hist)

In [ ]:
# ============ EVALUAR Y GRAFICAR ============
fig, axes = plt.subplots(len(CB_setpoints), 2, figsize=(14, 4 * len(CB_setpoints)))

T_fijo = 410.0  # Temperatura fija para variar CB

for i, CB_sp in enumerate(CB_setpoints):
    CB_hist, T_hist = evaluar_sp(trainer_eval, CB_sp, T_fijo)
    steps = np.arange(1, len(CB_hist) + 1)

    ax_CB = axes[i, 0]
    ax_CB.plot(steps, CB_hist, color='steelblue', linewidth=2, label='CB (PV)')
    ax_CB.axhline(CB_sp, color='red', linestyle='--', linewidth=1.5, label=f'SP={CB_sp}')
    ax_CB.axhspan(0.7, 1.15, alpha=0.1, color='green')
    ax_CB.set_title(f'CB — SP={CB_sp} mol/L')
    ax_CB.set_xlabel('Step')
    ax_CB.set_ylabel('CB (mol/L)')
    ax_CB.legend()
    ax_CB.grid(True, alpha=0.3)

    ax_T = axes[i, 1]
    ax_T.plot(steps, T_hist, color='orange', linewidth=2, label='T (PV)')
    ax_T.axhline(T_fijo, color='red', linestyle='--', linewidth=1.5, label=f'SP={T_fijo}K')
    ax_T.set_title(f'T — SP={T_fijo}K')
    ax_T.set_xlabel('Step')
    ax_T.set_ylabel('T (K)')
    ax_T.legend()
    ax_T.grid(True, alpha=0.3)

    print(f'CB_sp={CB_sp} → CB_final={CB_hist[-1]:.4f} (err={abs(CB_hist[-1]-CB_sp):.4f}) | '
          f'T_final={T_hist[-1]:.2f} (err={abs(T_hist[-1]-T_fijo):.2f}K)')

plt.suptitle('Evaluación AC (Transfer) — Distintos SP de CB', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('eval_transfer_CB_setpoints.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Test de Robustez: Perturbación en CA0

In [ ]:
# ============ PERTURBACIÓN: CA0 cambia de 5.1 a 5.5 mol/L ============
# Esto simula un cambio real en la alimentación del reactor.
# Los métodos clásicos NO se adaptan (PID fijo). El agente RL sí.

CB_sp, T_sp = 0.90, 407.0  # Mantener en SS

# --- RL con perturbación ---
state = trainer_eval.env.reset()[0]
trainer_eval.env.manipulable_setpoints = [CB_sp, T_sp]
trainer_eval.env._update_errors()
state = trainer_eval.env._get_observation()

CB_rl, T_rl = [], []
for step in range(50):
    if step == 10:
        reactor_eval2.set_disturbance(CA0=5.5)  # Perturbación en step 10
    action = trainer_eval.agent_ctrl.select_action(state, training=False)
    next_state, _, _, _, _ = trainer_eval.env.step(action)
    CB_rl.append(trainer_eval.env.manipulable_pvs[0])
    T_rl.append(trainer_eval.env.manipulable_pvs[1])
    state = next_state

# --- Graficar ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
steps = np.arange(1, len(CB_rl) + 1)

axes[0].plot(steps, CB_rl, color='steelblue', linewidth=2)
axes[0].axhline(CB_sp, color='red', linestyle='--')
axes[0].axvline(10, color='gray', linestyle=':', label='Perturbación CA0=5.5')
axes[0].set_title('CB ante perturbación')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, T_rl, color='orange', linewidth=2)
axes[1].axhline(T_sp, color='red', linestyle='--')
axes[1].axvline(10, color='gray', linestyle=':', label='Perturbación CA0=5.5')
axes[1].set_title('T ante perturbación')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Robustez del agente RL ante perturbación en CA0', fontweight='bold')
plt.tight_layout()
plt.savefig('robustez_perturbacion_CA0.png', dpi=150, bbox_inches='tight')
plt.show()